## Ejercicio 3: Inspección visual de un producto

Diseño de un agente inteligente que inspecciona una imagen binaria de 5 × 5 píxeles y determina el destino del producto según la cantidad de píxeles defectuosos detectados.

### Ficha PEAS

| Componente | Descripción |
|---|---|
| **P — Medida de desempeño** | Maximizar la detección correcta de productos defectuosos, reducir falsos rechazos y evitar que productos con fallas lleguen al cliente. |
| **E — Entorno** | Línea de producción, estación de inspección, productos fabricados, cámara y área de revisión manual. |
| **A — Acciones** | Aprobar el producto, enviarlo a revisión manual o rechazarlo. |
| **S — Percepciones** | Matriz NumPy de 5 × 5, donde `0` representa un píxel normal y `1` un píxel defectuoso. |

**Objetivo:** clasificar cada producto según la evidencia visual de defectos, permitiendo el paso de unidades conformes y separando las que necesitan revisión o rechazo.

### Justificación de las reglas

Se aplicarán las siguientes reglas de decisión:

- Si no se detectan píxeles defectuosos, el producto se aprueba.
- Si se detectan entre 1 y 3 píxeles defectuosos, el producto se envía a revisión manual.
- Si se detectan más de 3 píxeles defectuosos, el producto se rechaza.

Un producto sin evidencia visual de fallas puede continuar por la línea sin intervención adicional.
Entre uno y tres defectos existe incertidumbre suficiente para solicitar una revisión humana y evitar falsos rechazos.
Más de tres píxeles defectuosos indican una falla extendida y elevan el riesgo de entregar un producto no conforme.
Estas reglas equilibran la calidad del producto con el costo operativo de revisar cada unidad manualmente.

### Código

In [ ]:
import numpy as np


def agente_inspeccion(imagen):
    """
    Percepción (S): matriz 5 × 5 de NumPy
    (0 = píxel normal, 1 = píxel defectuoso).
    """
    if not isinstance(imagen, np.ndarray):
        raise TypeError("La imagen debe ser una matriz de NumPy.")
    if imagen.shape != (5, 5):
        raise ValueError("La imagen debe tener dimensiones 5 × 5.")
    if not np.all(np.isin(imagen, [0, 1])):
        raise ValueError("La imagen solo puede contener valores 0 y 1.")

    defectos = int(np.count_nonzero(imagen == 1))

    if defectos == 0:
        return "aprobar", "no se detectaron píxeles defectuosos"
    if defectos <= 3:
        return (
            "enviar a revisión manual",
            f"se detectaron {defectos} píxeles defectuosos",
        )
    return "rechazar", f"se detectaron {defectos} píxeles defectuosos"

### Simulación y pruebas

Se evalúan cinco matrices propias con 0, 1, 3, 4 y 8 píxeles defectuosos. Los casos con 0, 3 y 4 defectos comprueban los límites entre las tres acciones.

In [ ]:
imagen_sin_defectos = np.array([
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
])

imagen_un_defecto = np.array([
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
])

imagen_tres_defectos = np.array([
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1],
])

imagen_cuatro_defectos = np.array([
    [1, 1, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 1],
])

imagen_ocho_defectos = np.array([
    [1, 1, 1, 0, 0],
    [0, 1, 0, 1, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 1, 0],
    [0, 0, 0, 0, 0],
])

casos_prueba = [
    ("sin defectos", imagen_sin_defectos, "aprobar"),
    ("un defecto", imagen_un_defecto, "enviar a revisión manual"),
    ("límite de tres defectos", imagen_tres_defectos, "enviar a revisión manual"),
    ("cuatro defectos", imagen_cuatro_defectos, "rechazar"),
    ("ocho defectos", imagen_ocho_defectos, "rechazar"),
]

for numero, (descripcion, imagen, esperado) in enumerate(casos_prueba, start=1):
    accion, motivo = agente_inspeccion(imagen)
    defectos = int(np.count_nonzero(imagen == 1))
    assert accion == esperado, f"Caso {numero}: se esperaba '{esperado}' y se obtuvo '{accion}'"
    print(f"Caso {numero}: {descripcion}")
    print(f"  Píxeles defectuosos: {defectos}")
    print(f"  Decisión: {accion}")
    print(f"  Motivo: {motivo}\n")